# Load a bunch of annoying constants

In [ ]:
# Check whether it's okay to have the case study molecules in the HUVEC map training data.
# Check differences between drugsrceen v1_2 and v1_3
# Check differences between cross cell line v1_0 and v1_1

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
from dataclasses import dataclass
import json
from typing import Dict, Any

CROSS_CELL_LINE_BASE_PATH = Path(
    "/rxrx/data/valence/phenomics/cross_cell_line__brightfield__pretrain__v1_0"
)

RESERVATION_PATH = Path(
    "/rxrx/data/valence/internal_benchmarking/vcds1/v1_2_reservation.json"
)

CASE_STUDY_MOLECULES = Path("/rxrx/data/user/ali.denton/outgoing/candidates_for_case_studies.csv")

CROSS_CELL_LINE_OBS_PATH: Path = (
    CROSS_CELL_LINE_BASE_PATH
    / "cross_cell_line__brightfield__pretrain__v1_0_obs.parquet"
)
INTERNAL_BENCHMARKING_BASE_PATH = Path("/rxrx/data/valence/internal_benchmarking/")

COMPOUND_CONTROLS: set[str] = {
    "REC-0000058",
    "REC-0000069",
    "REC-0000098",
    "REC-0000219",
    "REC-0000370",
    "REC-0000375",
    "REC-0000391",
    "REC-0000493",
    "REC-0000584",
    "REC-0000590",
    "REC-0000671",
    "REC-0000706",
    "REC-0000737",
    "REC-0000741",
    "REC-0000773",
    "REC-0000842",
    "REC-0000854",
    "REC-0000867",
    "REC-0000889",
    "REC-0000959",
    "REC-0000970",
    "REC-0001029",
    "REC-0001106",
    "REC-0001148",
    "REC-0001183",
    "REC-0001214",
    "REC-0001246",
    "REC-0001249",
    "REC-0001250",
    "REC-0001258",
    "REC-0001289",
    "REC-0001290",
    "REC-0001610",
    "REC-0001619",
    "REC-0001655",
    "REC-0001721",
    "REC-0001730",
    "REC-0001744",
    "REC-0001907",
    "REC-0002007",
    "REC-0002082",
    "REC-0002101",
    "REC-0002178",
    "REC-0003420",
    "REC-0003935",
    "REC-0003980",
    "REC-0003982",
    "REC-0004011",
    "REC-0004273",
    "REC-0004419",
}


DRUGSCREEN_OBS_PATH: Path = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcds1"
    / "drugscreen__cell_paint__v1_3"
    / "drugscreen__cell_paint__v1_3_obs.parquet"
)
CROSS_CELL_LINE_TEST_OBS_PATH: Path = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcds1"
    / "cross_cell_line__brightfield__v1_1"
    / "cross_cell_line__brightfield__v1_1_obs.parquet"
)

COMMON_COLUMNS: list[str] = [
    "path",
    "image_type",
    "shape",
    "source",
    "image_type_id",
    "experiment",
    "experiment_id",
    "cell_condition",
    "cell_condition_id",
    "comp_id",
    "c_concs",
]

COMPOUND_PATTERN = r"^REC-\d+$"
TARGET_WELLS = 1000
TRAIN_FRACTION = 0.9
METADATA_SPLIT_SEED = 123456
OBS_SPLIT_SEED = 123456789

class Tokenizer:
    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = []

    def fit(self, df: pl.Series):
        unique_tokens: pl.Series = df.sort()
        for i, token in enumerate(unique_tokens):
            self.token_to_id[token] = i
            self.id_to_token.append(token)
        return self

    def transform(self, x):
        if isinstance(x, str):
            x = [x]
        return [self.token_to_id[token] for token in np.array(x).flatten()]
    
    def __len__(self):
        return len(self.id_to_token)

    def __call__(self, x):
        return self.transform(x)

def assign_random_split(
    df: pl.DataFrame, train_fraction: float, seed: int
) -> pl.DataFrame:
    """Assign a reproducible train/val split."""
    n_train = int(df.height * train_fraction)
    return df.with_columns(
        pl.when(pl.int_range(0, pl.len()).shuffle(seed=seed) < n_train)
        .then(pl.lit("train"))
        .otherwise(pl.lit("val"))
        .alias("split")
    )

# Images with zarrs that are known to be corrupted
SKIP_IMAGES: list[str] = pl.read_csv(
    "/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/zarr_scan_shards/skip_paths_all.csv"
)["path"].to_list()

@dataclass(frozen=True)
class SplitIndices:
    finetune: np.ndarray
    validation: np.ndarray
    test: np.ndarray

def load_vcb_split_indices(splits_path: Path) -> tuple[Dict[str, Any], SplitIndices]:
    """Load finetune/validation/test split indices."""
    with splits_path.open() as handle:
        splits_json = json.load(handle)
    first_fold = splits_json["folds"][0]
    split_indices = SplitIndices(
        finetune=np.array(first_fold["finetune"]),
        validation=np.array(first_fold["validation"]),
        test=np.array(first_fold["test"]),
    )
    return splits_json, split_indices

# Load the DRUGSCREEN splits
SPLITS_PATH: Path = (
    INTERNAL_BENCHMARKING_BASE_PATH
    / "vcb"
    / "splits"
    / "drugscreen__cell_paint__v1_2"
    / "split_compound_random__v1.json"
)
_, vcb_splits = load_vcb_split_indices(SPLITS_PATH)

In [41]:
df: pl.DataFrame = pl.read_parquet('/rxrx/data/microscopy/photosynthetic_metadata/95M_filtered_6chan_rp006.parquet')
old_shape: tuple[int, int] = df.shape
print(f"Original shape: {old_shape}")
df: pl.DataFrame = df.filter(~pl.col("path").is_null())
new_shape: tuple[int, int] = df.shape
print(f"Dropped {old_shape[0] - new_shape[0]} rows with null paths")
print(f"New shape: {new_shape}")
old_shape = new_shape
df = df.filter(pl.col("shape") == [2048, 2048, 6])
new_shape = df.shape
print(f"Dropped {old_shape[0] - new_shape[0]} rows with shape != [2048, 2048, 6]")
print(f"New shape: {new_shape}")

name1 = "bf_huvec_whole_genome_no_empty_v0.parquet"
name2 = "rptec_metadata_041625_no_96hr_no_empty.parquet"
base_path = "/rxrx/data/microscopy/photosynthetic_metadata/"
df_bf1 = pl.read_parquet(f"{base_path}{name1}")
df_bf2 = pl.read_parquet(f"{base_path}{name2}")
cols = list(set(df_bf1.columns).intersection(set(df_bf2.columns)))
df_bf = pl.concat([df_bf1[cols], df_bf2[cols]])

df_bf = df_bf.with_columns(
    pl.lit(False).alias("is_rp006"),
    pl.lit("brightfield_3channel").alias("image_type"),
    pl.lit([2048, 2048, 3]).alias("shape"),
)
sorted_cols = sorted(df_bf.columns)
df = pl.concat([df_bf[sorted_cols], df[sorted_cols]])
new_shape = df.shape
print(f"New shape: {new_shape}")
print(f"Added {old_shape[0]- df.shape[0]} rows from brightfield metadata")
df = df.with_columns(split=pl.lit("train")) # make everything train for now.

# Match things up with VCB data

df = df.rename({
        "path": "image_path",
        "experiment": "experiment_label",
        "address": "well_address"
    })

df = df.with_columns(
    concentration=pl.col("concentration") * 1000
)

Original shape: (92764542, 20)
Dropped 0 rows with null paths
New shape: (92764542, 20)
Dropped 33472396 rows with shape != [2048, 2048, 6]
New shape: (59292146, 20)
New shape: (75023432, 20)
Added 17741110 rows from brightfield metadata


In [42]:
def get_sampled_ids(df, cell_type, sample_fraction=0.1):
    # 1. Filter by cell type
    # 2. Transform rec_id (sort/join) to ensure consistent string representation
    # 3. Get unique values
    # 4. Sample
    return (
        df.filter(pl.col('cell_type') == cell_type)
          .select(pl.col('rec_id_string'))
          .to_series()
          .unique()
          .sample(fraction=sample_fraction, shuffle=True)
    )

df = df.with_columns(
    rec_id_string = pl.col('rec_id').list.sort().list.join('-')
)

arpe_valid_ids = get_sampled_ids(df, 'ARPE19')
rptec_valid_ids = get_sampled_ids(df, 'RPTEC')
huvec_valid_ids = get_sampled_ids(df, 'HUVEC_TNFalpha')
df = df.with_columns(
    pl.when(pl.col("rec_id_string").is_in(arpe_valid_ids) &  (pl.col("cell_type") == 'ARPE19')).then(pl.lit("valid_arpe19"))
      .when(pl.col("rec_id_string").is_in(rptec_valid_ids) & (pl.col("cell_type") == 'RPTEC')).then(pl.lit("valid_reptec"))
      .when(pl.col("rec_id_string").is_in(huvec_valid_ids) & (pl.col("cell_type") == 'HUVEC_TNFalpha')).then(pl.lit("valid_huvec_tnfalpha"))
      .otherwise(pl.col("split")) # or pl.lit("train")
      .alias("split")
)

/tmp/ipykernel_3113000/1802712642.py:21: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.with_columns(


In [43]:
df = df.with_row_index("row_idx") 
bf3_indices = df.filter(pl.col("image_type") == "brightfield_3channel").select("row_idx")
sampled_bf3_indices = bf3_indices.sample(fraction=0.002, shuffle=True) # just saving approximately 32 000 rows for FID

cp_indices = df.filter(pl.col("image_type") == "cellpaint").select("row_idx")
sampled_cp_indices = cp_indices.sample(fraction=0.0005, shuffle=True) # just saving approximately 32 000 rows for FID

df = df.with_columns(
    pl.when(pl.col("row_idx").is_in(sampled_bf3_indices["row_idx"]))
      .then(pl.lit("valid_bf3_iid"))
      .when(pl.col("row_idx").is_in(sampled_cp_indices["row_idx"]))
      .then(pl.lit("valid_cp_iid"))
      .otherwise(pl.col("split"))
      .alias("split")
).drop("row_idx") # Clean up the temporary index
df['split'].value_counts()

/tmp/ipykernel_3113000/2351498326.py:8: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  df = df.with_columns(


split,count
str,u32
"""valid_arpe19""",561539
"""valid_reptec""",802126
"""train""",73515110
"""valid_bf3_iid""",31462
"""valid_huvec_tnfalpha""",84264
"""valid_cp_iid""",28931


# Cross cell line

In [2]:
cross_cell_line_df_train: pl.DataFrame = (pl.read_parquet(str(CROSS_CELL_LINE_OBS_PATH)).with_columns(
        pl.lit("brightfield_3channel").alias("image_type"),
        pl.lit("cross_cell_line").alias("source"),
        pl.lit([2048, 2048, 3]).alias("shape"),
        pl.lit("train").alias("split"),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
)
cross_cell_line_df_test: pl.DataFrame = (pl.read_parquet(str(CROSS_CELL_LINE_TEST_OBS_PATH)).with_columns(
        pl.lit("brightfield_3channel").alias("image_type"),
        pl.lit("cross_cell_line").alias("source"),
        pl.lit([2048, 2048, 3]).alias("shape"),
        pl.lit("test").alias("split"),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )
)

cross_cell_line_df: pl.DataFrame = pl.concat([cross_cell_line_df_train, cross_cell_line_df_test[cross_cell_line_df_train.columns]])
print("Cell type diversity data (aka Charles' data)")
rec_id_tokenizer = Tokenizer().fit(cross_cell_line_df['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(cross_cell_line_df['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(cross_cell_line_df['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(cross_cell_line_df['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(cross_cell_line_df['experiment_label'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))

Cell type diversity data (aka Charles' data)
Unique rec_ids:  6655
Unique concentrations:  5
Unique cell_types:  20
Unique image_types:  1
Unique experiments:  198


In [3]:
# 1. Create the consistent string ID
cross_cell_line_df = cross_cell_line_df.with_columns(
    rec_id_string = pl.col('rec_id').list.sort().list.join('-')
)

# 2. Get a global set of unique IDs and sample 10% of them
# We do NOT group by cell_type here, so we get a global list.
sampled_ids = (
    cross_cell_line_df.filter(pl.col("cell_type") == "A_704") # use one cell type so the same rec_ids are used everywhere
    .select("rec_id_string")
    .unique()
    .sample(fraction=0.163, shuffle=True)
)

# 3. Assign the split column
# If the ID is in our sampled list, it is 'valid' everywhere it appears.
cross_cell_line_df = cross_cell_line_df.with_columns(
    pl.when(pl.col("rec_id_string").is_in(sampled_ids.get_column("rec_id_string")))
    .then(pl.lit("valid_cross_cell_line"))
    .otherwise(pl.lit("train"))
    .alias("split")
)

# 4. Check the results (Stats)
stats = (
    cross_cell_line_df
    .group_by(["cell_type", "split"])
    .agg(
        pl.col("rec_id_string").n_unique().alias("unique_count"),
        pl.len().alias("total_rows")
    )
    .sort(["cell_type", "split"])
)
with pl.Config(tbl_rows=100):
    print(stats)

/tmp/ipykernel_1467997/435604780.py:17: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  cross_cell_line_df = cross_cell_line_df.with_columns(


shape: (40, 4)
┌─────────────────────┬───────────────────────┬──────────────┬────────────┐
│ cell_type           ┆ split                 ┆ unique_count ┆ total_rows │
│ ---                 ┆ ---                   ┆ ---          ┆ ---        │
│ str                 ┆ str                   ┆ u32          ┆ u32        │
╞═════════════════════╪═══════════════════════╪══════════════╪════════════╡
│ AC16                ┆ train                 ┆ 6404         ┆ 82296      │
│ AC16                ┆ valid_cross_cell_line ┆ 250          ┆ 6000       │
│ A_704               ┆ train                 ┆ 1286         ┆ 38136      │
│ A_704               ┆ valid_cross_cell_line ┆ 250          ┆ 6000       │
│ BxPC3               ┆ train                 ┆ 1286         ┆ 38136      │
│ BxPC3               ┆ valid_cross_cell_line ┆ 250          ┆ 6000       │
│ HACAT               ┆ train                 ┆ 5126         ┆ 71256      │
│ HACAT               ┆ valid_cross_cell_line ┆ 250          ┆ 6000      

This parquet file (`cross_cell_line_v1.parquet`) is used for finetuning the model for the cross cell type task. Only the training examples are used and it is not used during pretraining.

In [5]:
cross_cell_line_df.write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/cross_cell_line_v1.parquet')

`cross_cell_line_inference.parquet` is where I'll save the parquet file that will be used for running inference with the model. When I originally saved it I forgot to include controls, so I've now also saved `cross_cell_line_controls.parquet`. Code for both is below

In [ ]:
exps_and_cell_types = cross_cell_line_df.filter(pl.col('split').str.contains('valid_'))['experiment_label', 'cell_type'].unique()
cross_cell_line_df = cross_cell_line_df.with_columns(is_negative_control = pl.col('rec_id_string') == "")
exp_controls = []
for cell_type in exps_and_cell_types['cell_type'].unique():
    exp_subset = cross_cell_line_df.filter(
        (pl.col('split').str.contains('valid_'))
        & (pl.col('cell_type') == cell_type))['experiment_label'].unique()
    exp_controls.append(cross_cell_line_df.filter(
        (pl.col('is_negative_control') ) & 
        (pl.col('cell_type') == cell_type) & 
        (pl.col('experiment_label').is_in(exp_subset)) 
        )
    )
exp_controls = pl.concat(exp_controls)
exp_controls.write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/big-img/metadata/cross_cell_line_controls.parquet')
cross_cell_line_df.filter(pl.col('split').str.contains('valid_')).write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/big-img/metadata/cross_cell_line_inference.parquet')

# Drugscreen

This is almost exactly the same as the steps above except we use the splits to define train / test / valid.

**Only potential gotcha:** see how I'm adding an id column and check its okay

In [7]:
drugscreen_obs = pl.read_parquet(str(DRUGSCREEN_OBS_PATH)).with_columns(
    id=pl.arange(0, pl.len()), # NB: I'd appreciate a second set of eyes on this. It assumes that the rows in the splits correspond to the rows in the VCB obs data.
).with_columns(
        pl.lit("cellpaint").alias("image_type"),
        pl.lit("drugscreen").alias("source"),
        pl.lit([2048, 2048, 6]).alias("shape"),
        split = pl.when(pl.col("id").is_in(vcb_splits.finetune))
        .then(pl.lit("train"))
            .when(pl.col("id").is_in(vcb_splits.validation))
        .then(pl.lit("valid_drugscreen"))
        .when(pl.col("id").is_in(vcb_splits.test))
        .then(pl.lit("test_drugscreen"))
        .otherwise(pl.lit(None)),
        rec_id=pl.col("perturbations").list.eval(
            pl.element().struct.field("source_id")
        ),
        concentration=pl.col("perturbations").list.eval(
            pl.element().struct.field("concentration").cast(pl.Float64).cast(pl.String)
        ),
    )

print("DRUGSCREEN data")
rec_id_tokenizer = Tokenizer().fit(drugscreen_obs['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(drugscreen_obs['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(drugscreen_obs['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(drugscreen_obs['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(drugscreen_obs['experiment_label'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))


DRUGSCREEN data
Unique rec_ids:  2712
Unique concentrations:  22
Unique cell_types:  2
Unique image_types:  1
Unique experiments:  56


# Please Check!

This part is the only really annoying bit - we want to add "one plate of data" to the training set to enable generalization to new perturbations.

I'll be honest - I'm a little unsure of how much is reasonable here because in practice if this worked well, we'd do a whole geneome knockout with some modest set of compounds on each gene knockout - so unlike the cell type diversity task, I don't think you'd do 1 plate per gene (that's 20000 plates!)

In [8]:
# Masks for finding controls
is_neg_or_base = (pl.col("is_negative_control")) | (pl.col("is_base_state"))
is_comp_control = (
    pl.col("rec_id")
    .list.eval(pl.element().is_in(COMPOUND_CONTROLS))
    .list.any()
)

any_control_mask = is_neg_or_base | is_comp_control

drugscreen_obs = drugscreen_obs.with_columns(no_comp_rec_id=pl.col("perturbations")
        .list.eval(pl.element().struct.field("source_id"))
        .list.eval(
            pl.element().filter(pl.element().str.contains(COMPOUND_PATTERN).not_())
        )).with_columns(
    cell_condition=pl.format(
        "{}_{}",
        pl.col("cell_type"),
        pl.col("no_comp_rec_id").list.sort().list.join("-"),
    )
)

# 1. Batch Matching (Existing Training Data)
    # Find (Experiment, CellType) pairs in existing train set
train_context = (
    drugscreen_obs.filter(pl.col("split") == "train")
    .select(["experiment_label", "cell_type"])
    .unique()
)
# Find controls in those same batches
train_matched_controls = (
    drugscreen_obs.join(train_context, on=["experiment_label", "cell_type"], how="inner")
    .filter(is_neg_or_base)
    .select("id")
)

# 2. Virtual Plate (Val/Test Data)
# Find cell_conditions in val/test
val_test_conditions = (
    drugscreen_obs.filter(pl.col("split").is_in(["val", "test"]))
    .select("cell_condition")
    .unique()
)
# Filter candidates (Base + Neg + Positive Controls) and match condition
virtual_plate_candidates = drugscreen_obs.filter(any_control_mask).join(
    val_test_conditions, on="cell_condition", how="inner"
)
# Rank plates by density
plate_rankings = (
    virtual_plate_candidates.group_by(
        ["cell_condition", "experiment_label", "experiment_plate_number"]
    )
    .len(name="density")
    .sort(["cell_condition", "density"], descending=True)
)
# Pick best plate per condition
best_plates = plate_rankings.group_by("cell_condition", maintain_order=True).head(1)
# Get IDs
virtual_plate_controls = virtual_plate_candidates.join(
    best_plates,
    on=["cell_condition", "experiment_label", "experiment_plate_number"],
    how="inner",
).select("id")

print(f"Virtual plate controls adds {virtual_plate_controls.shape[0]} rows to the training set")

# 3. Update Split Column
ids_to_add_to_train = train_matched_controls.vstack(virtual_plate_controls).unique()

drugscreen_obs = drugscreen_obs.with_columns(
    split=pl.when(pl.col("id").is_in(ids_to_add_to_train["id"]))
    .then(pl.lit("train"))
    .otherwise(pl.col("split"))
)

Virtual plate controls adds 0 rows to the training set


/tmp/ipykernel_3722426/3815731992.py:70: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  drugscreen_obs = drugscreen_obs.with_columns(


In [16]:
drugscreen_obs.filter(is_neg_or_base).write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/big-img/metadata/drugscreen_controls.parquet')

# Active Learning Data

In [48]:
import zarr
import pickle
import polars as pl

def get_shape(path) -> tuple[int, int, int]:
    if len(path) > 0:
        return tuple(zarr.open(str(path), mode='r').shape)
    else:
        return None

al = pl.read_parquet("/mnt/ps/home/CORP/jason.hartford/pythondev/photosynthetic/notebooks/AL_metadata.parquet")
al = al.with_columns(
    pl.lit("train").alias("split"),
    n_perts = pl.col("rec_id").list.len(),
    rec_id_string = pl.col('rec_id').list.sort().list.join('-'),
    pert_type_string = pl.col("perturbation_type").list.sort().list.join('-')
).with_row_index("row_id")
al = al.with_columns(
        pl.when(pl.col("n_perts") > 1)
        .then(pl.lit("valid_al"))
        .otherwise(pl.lit("train"))
        .alias("split")
    )
pert_types = ["COMPOUND-GUIDE", "COMPOUND-COMPOUND", "GUIDE-GUIDE"]
for pert_type in pert_types:
    train_ids = al.filter((pl.col("n_perts") > 1) & (pl.col("pert_type_string") == pert_type))['rec_id_string'].unique().sample(n=250, shuffle=True)
    al = al.with_columns(
        pl.when(pl.col("rec_id_string").is_in(train_ids))
        .then(pl.lit("train"))
        .otherwise(pl.col("split"))
        .alias("split")
    )
# see code below for how this was generated
shapes = pickle.load(open("shapes.pkl", "rb"))
al = al.with_columns(
    shape = pl.Series(shapes)
)

al = al.filter(pl.col("shape") == [2048, 2048, 3]).with_columns(
    pl.lit("brightfield_3channel").alias("image_type"),
    pl.lit("active_learning").alias("source"),
    concentration = (pl.col("concentration") * 1000).list.eval(
        pl.element().cast(pl.String),
    )
).rename({
    "path": "image_path",
    "experiment": "experiment_label",
    "address": "well_address"
}).with_columns(
    pl.concat_str(['plate', 'order_id' ,'read_id'], separator='_').alias('plate_order_read_id'),
)

stats = (
    al
    .group_by(["n_perts", "pert_type_string", "split"])
    .agg(
        pl.col("rec_id_string").n_unique().alias("unique_count"),
        pl.len().alias("total_rows")
    )
    .sort(["n_perts", "pert_type_string", "split"])
)
with pl.Config(tbl_rows=100):
    print(stats)

/tmp/ipykernel_3113000/2891109281.py:27: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  al = al.with_columns(


shape: (9, 5)
┌─────────┬───────────────────┬──────────┬──────────────┬────────────┐
│ n_perts ┆ pert_type_string  ┆ split    ┆ unique_count ┆ total_rows │
│ ---     ┆ ---               ┆ ---      ┆ ---          ┆ ---        │
│ u32     ┆ str               ┆ str      ┆ u32          ┆ u32        │
╞═════════╪═══════════════════╪══════════╪══════════════╪════════════╡
│ 0       ┆                   ┆ train    ┆ 1            ┆ 86098      │
│ 1       ┆ COMPOUND          ┆ train    ┆ 50           ┆ 10801      │
│ 1       ┆ GUIDE             ┆ train    ┆ 1421         ┆ 112863     │
│ 2       ┆ COMPOUND-COMPOUND ┆ train    ┆ 250          ┆ 16714      │
│ 2       ┆ COMPOUND-COMPOUND ┆ valid_al ┆ 1025         ┆ 67118      │
│ 2       ┆ COMPOUND-GUIDE    ┆ train    ┆ 221          ┆ 5335       │
│ 2       ┆ COMPOUND-GUIDE    ┆ valid_al ┆ 7279         ┆ 174604     │
│ 2       ┆ GUIDE-GUIDE       ┆ train    ┆ 220          ┆ 2445       │
│ 2       ┆ GUIDE-GUIDE       ┆ valid_al ┆ 47533        ┆ 53430

In [49]:
# import pickle
# shapes = []
# for i in tqdm.tqdm(range(al.shape[0])):
#     try:
#         shapes.append(get_shape(al[i]["path"][0]))
#     except Exception as e:
#         print(e)
#         shapes.append(None)
# pickle.dump(shapes, open("shapes.pkl", "wb"))

#al.write_parquet("/mnt/ps/home/CORP/jason.hartford/pythondev/photosynthetic/notebooks/AL_metadata.parquet")


# Now lets merge it all together

In [50]:
cross_cell_line_df['split'].value_counts()

split,count
str,u32
"""valid_cross_cell_line""",118688
"""train""",964128


In [51]:
cols

['experiment',
 'site',
 'order_id',
 'perturbation',
 'perturbation_type',
 'batch_id',
 'read_id',
 'concentration',
 'address',
 'gene',
 'plate',
 'rec_id',
 'cell_type',
 'control',
 '__index_level_0__',
 'split',
 'path']

In [52]:
df = df.with_columns(
    pl.concat_str(['plate', 'order_id' ,'read_id'], separator='_').alias('plate_order_read_id'),
    pl.lit("pretraining").alias("source"),
    concentration=pl.col('concentration').list.eval(
    pl.element().cast(pl.String),
))
cols = sorted(list(set(df.columns).intersection(set(drugscreen_obs.columns))))
combined = pl.concat([df[cols], drugscreen_obs[cols], cross_cell_line_df[cols], al[cols]])
combined = combined.filter(~pl.col("image_path").is_in(SKIP_IMAGES)) # skip images that are known to be corrupted

In [53]:
combined.with_columns(pl.concat_str(['plate_order_read_id', 'well_address'], separator=':').alias('obs_id')).write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/pretraining_v5.parquet')

In [55]:
print("Combined data")
rec_id_tokenizer = Tokenizer().fit(combined['rec_id'].explode().unique())
concentration_tokenizer = Tokenizer().fit(combined['concentration'].explode().unique())
cell_type_tokenizer = Tokenizer().fit(combined['cell_type'].unique())
image_type_tokenizer = Tokenizer().fit(combined['image_type'].unique())
experiment_tokenizer = Tokenizer().fit(combined['experiment_label'].unique())
well_tokenizer = Tokenizer().fit(combined['well_address'].unique())

print("Unique rec_ids: ", len(rec_id_tokenizer))
print("Unique concentrations: ", len(concentration_tokenizer))
print("Unique cell_types: ", len(cell_type_tokenizer))
print("Unique image_types: ", len(image_type_tokenizer))
print("Unique experiments: ", len(experiment_tokenizer))
print("Unique well_addresses: ", len(well_tokenizer))


Combined data


Unique rec_ids:  613486
Unique concentrations:  329
Unique cell_types:  43
Unique image_types:  4
Unique experiments:  8846
Unique well_addresses:  1531


# Testing that the parquet file is correct:

In [1]:
import polars as pl
combined_v5 = pl.read_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/pretraining_v5.parquet')

In [2]:
with pl.Config(tbl_rows=100):
    print(combined_v5.filter(pl.col('split').str.contains('valid_'))['split'].value_counts())
combined_v5.filter(pl.col('split').is_in(['valid_al', 'valid_drugscreen', 'valid_cross_cell_line'])).write_parquet('/mnt/ps/home/CORP/jason.hartford/project/big-x/joint-model/metadata/inference_v5.parquet')

shape: (8, 2)
┌───────────────────────┬────────┐
│ split                 ┆ count  │
│ ---                   ┆ ---    │
│ str                   ┆ u32    │
╞═══════════════════════╪════════╡
│ valid_drugscreen      ┆ 40544  │
│ valid_cross_cell_line ┆ 118688 │
│ valid_reptec          ┆ 802126 │
│ valid_huvec_tnfalpha  ┆ 84263  │
│ valid_cp_iid          ┆ 28931  │
│ valid_arpe19          ┆ 561539 │
│ valid_al              ┆ 776025 │
│ valid_bf3_iid         ┆ 31462  │
└───────────────────────┴────────┘


In [3]:
combined_v5.filter(pl.col('split').is_in(['valid_drugscreen']))

cell_type,concentration,experiment_label,image_path,image_type,order_id,plate_order_read_id,read_id,rec_id,shape,source,split,well_address,obs_id
str,list[str],str,str,str,i64,str,i64,list[str],list[i64],str,str,str,str
"""HUVEC""","[""100.0"", ""10.0""]","""PPOX-CustomPhenoMap-H-c""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5230,"""189399_5230_181""",181,"[""REC-GRNA-0156496"", ""REC-0001894""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""AA26""","""189399_5230_181:AA26"""
"""HUVEC""","[""25.0"", ""100.0""]","""PPOX-CustomPhenoMap-H-c""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5230,"""189399_5230_181""",181,"[""REC-0000874"", ""REC-GRNA-0156496""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""AD29""","""189399_5230_181:AD29"""
"""HUVEC""","[""100.0"", ""2.5""]","""PPOX-CustomPhenoMap-H-c""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5230,"""189399_5230_181""",181,"[""REC-GRNA-0156496"", ""REC-0001894""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""AE29""","""189399_5230_181:AE29"""
"""HUVEC""","[""100.0"", ""2.5""]","""PPOX-CustomPhenoMap-H-c""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5230,"""189399_5230_181""",181,"[""REC-GRNA-0156496"", ""REC-0000874""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""E22""","""189399_5230_181:E22"""
"""HUVEC""","[""10000.0"", ""100.0""]","""PPOX-CustomPhenoMap-H-c""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5230,"""189399_5230_181""",181,"[""REC-0000874"", ""REC-GRNA-0156496""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""E45""","""189399_5230_181:E45"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""HUVEC""","[""100.0"", ""25.0""]","""CNOT3-CustomPhenoMap-H-a""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5272,"""179370_5272_70""",70,"[""REC-GRNA-0017928"", ""REC-0152537""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""V19""","""179370_5272_70:V19"""
"""HUVEC""","[""100.0"", ""100.0""]","""CNOT3-CustomPhenoMap-H-a""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5272,"""179370_5272_70""",70,"[""REC-GRNA-0017928"", ""REC-0161976""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""W04""","""179370_5272_70:W04"""
"""HUVEC""","[""100.0"", ""10.0""]","""CNOT3-CustomPhenoMap-H-a""","""/rxrx/data/microscopy/zarr_512…","""cellpaint""",5272,"""179370_5272_70""",70,"[""REC-GRNA-0017928"", ""REC-0083357""]","[2048, 2048, 6]","""drugscreen""","""valid_drugscreen""","""Y29""","""179370_5272_70:Y29"""


In [56]:
combined = combined.with_columns(
    rec_id_string = pl.col('rec_id').list.sort().list.join('-')
)
(combined.filter(pl.col('cell_type') == 'AC16').group_by(["cell_type", "split"]).agg(
        pl.col("rec_id_string").n_unique().alias("unique_count"),
        pl.len().alias("total_rows")
    )
    .sort(["cell_type", "split"])
)

cell_type,split,unique_count,total_rows
str,str,u32,u32
"""AC16""","""train""",6500,97067
"""AC16""","""valid_cp_iid""",12,13
"""AC16""","""valid_cross_cell_line""",250,6000


In [58]:
with pl.Config(tbl_rows=100):
    print(combined.filter(pl.col('split').str.contains('valid_'))['split'].value_counts())

shape: (8, 2)
┌───────────────────────┬────────┐
│ split                 ┆ count  │
│ ---                   ┆ ---    │
│ str                   ┆ u32    │
╞═══════════════════════╪════════╡
│ valid_cross_cell_line ┆ 118688 │
│ valid_cp_iid          ┆ 28931  │
│ valid_drugscreen      ┆ 40544  │
│ valid_huvec_tnfalpha  ┆ 84263  │
│ valid_al              ┆ 776025 │
│ valid_bf3_iid         ┆ 31462  │
│ valid_arpe19          ┆ 561539 │
│ valid_reptec          ┆ 802126 │
└───────────────────────┴────────┘
